# Lecture 10 — Retrieval-Augmented Generation (RAG)
## A Step-by-Step Implementation

This notebook builds a complete RAG pipeline from scratch using only local tools — no API keys needed.

| Component | Tool |
|-----------|------|
| Corpus | 5 plain-text documents in `docs/` |
| Embedding model | `nomic-embed-text` via Ollama |
| Vector store | ChromaDB (persistent, saved to `chroma_db/`) |
| Generator | `llama3.2` via Ollama |

**Prerequisites** — pull the two required Ollama models before running:
```bash
ollama pull llama3.2
ollama pull nomic-embed-text
```

## RAG Architecture

```
OFFLINE (once)
  Documents
      │  Step 1: chunk
      ▼
  Text chunks
      │  Step 2: embed
      ▼
  Dense vectors  ──►  ChromaDB (vector index)

ONLINE (per query)
  User query
      │  embed query
      ▼
  Query vector  ──►  ChromaDB nearest-neighbour search
                           │  top-k chunks
                           ▼
               Augmented prompt = context + question
                           │  generate
                           ▼
                      LLM answer
```

Two key ideas:
- **Parametric memory**: knowledge baked into model weights during pretraining.
- **Non-parametric memory**: documents retrieved at runtime from an external index.

RAG combines both. The LLM reasons; the index remembers.

---
## 0. Prerequisites & Imports

In [ ]:
import subprocess, sys

def check_ollama():
    try:
        r = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=5)
        if r.returncode == 0:
            print("Ollama is running. Available models:")
            print(r.stdout)
        else:
            print("Ollama error:", r.stderr)
    except FileNotFoundError:
        print("Ollama not found — install from https://ollama.com")
        sys.exit(1)
    except subprocess.TimeoutExpired:
        print("Ollama timed out — start it with: ollama serve")

check_ollama()

In [ ]:
from pathlib import Path
from typing import List, Tuple

import numpy as np
import chromadb
import ollama
from tqdm.notebook import tqdm

print("All imports OK.")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
EMBED_MODEL   = "nomic-embed-text"   # Ollama embedding model
GEN_MODEL     = "llama3.2"           # Ollama generation model
DOCS_DIR      = Path("docs")         # corpus directory
CHROMA_DIR    = Path("chroma_db")    # ChromaDB persistence path
COLLECTION    = "lecture10_rag"      # ChromaDB collection name
CHUNK_SIZE    = 400                  # characters per chunk
CHUNK_OVERLAP = 50                   # character overlap between adjacent chunks
TOP_K         = 3                    # chunks to retrieve per query
# ──────────────────────────────────────────────────────────────────────────────
print(f"Embed model : {EMBED_MODEL}")
print(f"Gen model   : {GEN_MODEL}")
print(f"Chunk size  : {CHUNK_SIZE} chars  overlap={CHUNK_OVERLAP}  top_k={TOP_K}")

---
## Step 1 — Load the Document Corpus

We have five short documents in `docs/`, each covering a different NLP/LLM topic:

| File | Topic |
|------|-------|
| `01_transformers.txt` | Transformer architecture |
| `02_attention.txt` | Self-attention mechanism |
| `03_rag.txt` | Retrieval-Augmented Generation |
| `04_pretraining.txt` | LLM pretraining |
| `05_fine_tuning.txt` | Fine-tuning and RLHF |

In a real system this step would use a PDF parser, a web scraper, or a database connector.
For now we just read plain text files.

In [ ]:
def load_documents(docs_dir: Path) -> List[dict]:
    docs = []
    for path in sorted(docs_dir.glob("*.txt")):
        text = path.read_text(encoding="utf-8")
        docs.append({"filename": path.name, "text": text})
    return docs

documents = load_documents(DOCS_DIR)
print(f"Loaded {len(documents)} documents:")
for doc in documents:
    print(f"  {doc['filename']}  ({len(doc['text'])} chars)")

In [ ]:
# Display the beginning of the first document
first = documents[0]
print(f"=== {first['filename']} (first 500 chars) ===")
print(first["text"][:500], "...")

---
## Step 2 — Split Documents into Chunks

Embedding models have limited token budgets (e.g. 512 or 8192 tokens). We split each document
into **overlapping fixed-size chunks** so that:
- No chunk exceeds the model limit.
- The overlap keeps sentences from being cut at a boundary and lost.

```
Document text:  [ . . . . . . . . . . . . . . . . . . . . . . . . . ]

chunk 0:        [──────── 400 chars ────────]
chunk 1:               [──────── 400 chars ────────]
chunk 2:                      [──────── 400 chars ────────]
                        ◄──►
                       50-char overlap
```

**Alternative chunkers** used in production: sentence-boundary splitting,
recursive text splitting (LangChain), semantic chunking.

In [ ]:
def chunk_text(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP,
) -> List[str]:
    """Split text into overlapping fixed-size character windows."""
    chunks, start = [], 0
    step = chunk_size - overlap
    while start < len(text):
        chunk = text[start : start + chunk_size].strip()
        if len(chunk) > 30:          # discard tiny trailing fragments
            chunks.append(chunk)
        start += step
    return chunks

In [ ]:
all_chunks: List[str] = []
chunk_metadata: List[dict] = []

for doc in documents:
    chunks = chunk_text(doc["text"])
    for idx, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({"source": doc["filename"], "chunk_index": idx})

print(f"Total chunks : {len(all_chunks)}")
print()
print(f"Sample chunk from '{chunk_metadata[0]['source']}' (index 0):")
print("-" * 60)
print(all_chunks[0])

---
## Step 3 — Embed Chunks with `nomic-embed-text`

An **embedding** converts a piece of text into a dense real-valued vector.
Texts with similar meaning end up close together in this vector space;
unrelated texts are far apart.

`nomic-embed-text` produces **768-dimensional** vectors and runs fully locally
via Ollama — no internet or API key required.

We embed every chunk once (offline). Later, we embed each incoming
query and search for the most similar chunk vectors.

In [ ]:
def embed_text(text: str) -> List[float]:
    """Return the embedding vector for a single text string."""
    response = ollama.embed(model=EMBED_MODEL, input=text)
    return response["embeddings"][0]

In [ ]:
print(f"Embedding {len(all_chunks)} chunks with '{EMBED_MODEL}' ...")
embeddings: List[List[float]] = [
    embed_text(chunk) for chunk in tqdm(all_chunks, desc="Embedding")
]
print("Done.")

In [ ]:
emb = np.array(embeddings)
print(f"Embedding matrix shape: {emb.shape}")
print(f"  {emb.shape[0]} chunks  x  {emb.shape[1]} dimensions")

# Cosine similarity helper
def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Sanity check: two consecutive chunks from the same document should be similar
sim_same = cosine_sim(emb[0], emb[1])
# Two chunks from different documents should be less similar
# Find an index from a different doc
diff_idx = next(i for i, m in enumerate(chunk_metadata) if m["source"] != chunk_metadata[0]["source"])
sim_diff = cosine_sim(emb[0], emb[diff_idx])

print(f"\nCosine similarity — same doc (chunks 0 & 1)         : {sim_same:.3f}")
print(f"Cosine similarity — different docs (chunk 0 & {diff_idx:2d})  : {sim_diff:.3f}")
print("(Higher = more similar. Same-doc chunks should score higher.)")

---
## Step 4 — Build the Vector Index with ChromaDB

ChromaDB is a pure-Python vector database that:
- Accepts documents, embeddings, and metadata.
- Persists everything to disk (in `chroma_db/`).
- Supports cosine and L2 distance for nearest-neighbour search.

We create one **collection** (like a table) and add all chunks plus their
precomputed embeddings. On subsequent runs you can skip re-embedding
and just load the persisted collection.

In [ ]:
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Drop existing collection so we get a clean index on re-run
try:
    client.delete_collection(COLLECTION)
    print(f"Deleted existing collection '{COLLECTION}'.")
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION,
    metadata={"hnsw:space": "cosine"},   # use cosine distance
)
print(f"Created collection '{COLLECTION}'.")

In [ ]:
ids = [f"chunk_{i}" for i in range(len(all_chunks))]

collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=all_chunks,
    metadatas=chunk_metadata,
)

print(f"Indexed {collection.count()} chunks into ChromaDB.")
print(f"Persisted to: {CHROMA_DIR.resolve()}")

---
## Step 5 — Retrieve Relevant Chunks

Retrieval is a **semantic nearest-neighbour search**:

1. Embed the user query with the same model used for indexing.
2. Search ChromaDB for the `k` vectors closest to the query vector.
3. Return the corresponding text chunks.

> **Why must we use the same embedding model for queries and documents?**  
> The embedding space is model-specific. Mixing models would make distances meaningless.

In [ ]:
def retrieve(
    query: str,
    k: int = TOP_K,
) -> List[Tuple[str, dict, float]]:
    """
    Embed query and return the top-k most relevant (chunk, metadata, distance) tuples.
    ChromaDB cosine distance: 0.0 = identical, 2.0 = opposite direction.
    """
    q_emb = embed_text(query)
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    chunks    = results["documents"][0]
    metas     = results["metadatas"][0]
    distances = results["distances"][0]
    return list(zip(chunks, metas, distances))

In [ ]:
test_query = "How does the attention mechanism work in transformers?"
print(f"Query: {test_query!r}\n")

retrieved = retrieve(test_query)
for rank, (chunk, meta, dist) in enumerate(retrieved, start=1):
    print(f"── Rank {rank}  source={meta['source']}  chunk#{meta['chunk_index']}  distance={dist:.4f}")
    print(chunk[:300])
    print()

---
## Step 6 — Generate an Answer with Augmented Context

We build an **augmented prompt** that places the retrieved chunks as context
before the user's question.  The LLM is instructed to answer **only** from
the provided context, which:
- grounds the answer in real documents,
- reduces hallucinations,
- makes answers traceable to a source.

The prompt structure:
```
System instruction

CONTEXT:
<chunk 1>
---
<chunk 2>
---
<chunk 3>

QUESTION:
<user query>

ANSWER:
```

In [ ]:
def build_prompt(query: str, context_chunks: List[str]) -> str:
    separator = "\n\n---\n\n"
    ctx = separator.join(context_chunks)
    parts = [
        "You are a helpful NLP assistant.",
        "Answer the question using ONLY the context provided below.",
        "If the answer cannot be found in the context, reply exactly: I don't know based on the provided documents.",
        "",
        "CONTEXT:",
        ctx,
        "",
        "QUESTION:",
        query,
        "",
        "ANSWER:",
    ]
    return "\n".join(parts)


def generate(prompt: str) -> str:
    response = ollama.chat(
        model=GEN_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response["message"]["content"]

In [ ]:
# Show the first 600 chars of the augmented prompt
chunks_only = [c for c, _, _ in retrieved]
prompt = build_prompt(test_query, chunks_only)

print("=== Augmented Prompt (first 600 chars) ===")
print(prompt[:600], "...\n")

print("=== Generated Answer ===")
answer = generate(prompt)
print(answer)

---
## Step 7 — The Full RAG Pipeline

Everything above, composed into a single `rag_query()` function.

In [ ]:
def rag_query(query: str, k: int = TOP_K, verbose: bool = False) -> str:
    # 1. Retrieve relevant chunks
    hits = retrieve(query, k=k)
    chunks = [c for c, _, _ in hits]

    if verbose:
        print(f"Retrieved {len(chunks)} chunks:")
        for i, (c, m, d) in enumerate(hits, 1):
            print(f"  [{i}] {m['source']}  dist={d:.4f}  '{c[:60]}...'")
        print()

    # 2. Build augmented prompt and generate
    prompt = build_prompt(query, chunks)
    return generate(prompt)

In [ ]:
questions = [
    "What is the role of the feed-forward network in a transformer layer?",
    "How does RLHF improve language model behaviour?",
    "What problem does RAG solve that fine-tuning alone cannot?",
    "What are the Chinchilla scaling laws?",
    "What is LoRA and why is it useful?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_query(q)}")
    print("-" * 70)

---
## Step 8 — RAG vs. No-RAG Comparison

Let's compare a RAG-grounded answer with a baseline that sends the question
directly to the LLM — no retrieved context, just parametric memory.

**Expected observations:**
- The RAG answer stays close to our documents (may quote wording from them).
- The no-RAG answer draws on the model's training data, which may differ from or contradict our corpus.
- For topics not in our corpus, both answers come only from parametric memory and will be similar.

In [ ]:
def no_rag_query(query: str) -> str:
    response = ollama.chat(
        model=GEN_MODEL,
        messages=[{"role": "user", "content": query}],
    )
    return response["message"]["content"]


def compare(query: str) -> None:
    print("QUERY:", query)
    print("=" * 70)

    print("[WITH RAG] — grounded in our 5 documents:")
    print(rag_query(query, verbose=True))

    print()
    print("[WITHOUT RAG] — model parametric memory only:")
    print(no_rag_query(query))
    print("=" * 70)

In [ ]:
compare("What is retrieval-augmented generation and why is it useful?")

In [ ]:
# Try a question whose answer IS in our corpus but is niche enough
# that the model might hallucinate without retrieval
compare("What does the Chinchilla paper say about the optimal number of tokens per parameter?")

---
## Summary

| Step | What we did | Tool |
|------|------------|------|
| 1 — Load | Read `.txt` files into memory | Python `pathlib` |
| 2 — Chunk | Split into overlapping 400-char windows | custom `chunk_text()` |
| 3 — Embed | Convert each chunk to a 768-dim vector | `nomic-embed-text` (Ollama) |
| 4 — Index | Store vectors on disk | ChromaDB |
| 5 — Retrieve | Nearest-neighbour search on the query vector | ChromaDB |
| 6 — Generate | LLM answers from the augmented prompt | `llama3.2` (Ollama) |

### Key takeaways

- RAG separates **what to say** (retrieved docs) from **how to say it** (LLM).
- You can update the knowledge base at any time without retraining.
- Retrieval quality sets an upper bound on answer quality — garbage in, garbage out.
- The same embedding model **must** be used at index time and query time.

### Next steps / exercises

1. Add more documents to `docs/` and observe how retrieval quality changes.
2. Change `CHUNK_SIZE` and `CHUNK_OVERLAP` and measure the effect on the answers.
3. Swap the embedding model (e.g. `mxbai-embed-large`) and compare retrieval results.
4. Implement a **re-ranker**: after retrieving top-10, re-rank with a cross-encoder and keep top-3.
5. Add **query expansion**: ask the LLM to rewrite the query before retrieval.